## train_with_gan

In [1]:
import pickle
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ==== 1. load original data ====
with open('/kaggle/input/fsdfsd/train.pickle', 'rb') as f:
    train_data = pickle.load(f)
x_train = np.stack(train_data['sensor_data'])
y_train = np.array(train_data['label'])

# ==== 2. load GAN-generated data ====
with open('/kaggle/input/fsdfsd/generated_data.pickle', 'rb') as f:
    gan_data = pickle.load(f)
x_gan = np.array([item['sensor_data'] for item in gan_data])
y_gan = np.array([item['label'] for item in gan_data])

# ==== 3. load test set ====
with open('/kaggle/input/fsdfsd/test.pickle', 'rb') as f:
    test_data = pickle.load(f)
x_test = np.stack(test_data['sensor_data'])
y_test = np.array(test_data['label'])

# ==== 4. only keep categories which exist in test set ====
test_classes = set(np.unique(y_test))
mask_train = np.array([y in test_classes for y in y_train])
mask_gan = np.array([y in test_classes for y in y_gan])

x_train = x_train[mask_train]
y_train = y_train[mask_train]
x_gan = x_gan[mask_gan]
y_gan = y_gan[mask_gan]

# LabelEncoder only encodes the categories involved in the test set
all_classes = sorted(list(test_classes))
le = LabelEncoder()
le.fit(all_classes)
y_train = le.transform(y_train)
y_gan = le.transform(y_gan)
y_test = le.transform(y_test)

def print_label_stats(name, y):
    unique, counts = np.unique(y, return_counts=True)
    print(f"{name} label dist:", dict(zip(unique, counts)))
print_label_stats("Train", y_train)
print_label_stats("Test", y_test)
print_label_stats("GAN", y_gan)

# ==== 5. Merge GAN data ====
x_train = np.concatenate([x_train, x_gan], axis=0)
y_train = np.concatenate([y_train, y_gan], axis=0)

# ==== 6. Upsampling to achieve class balance ====
num_classes = len(all_classes)
max_count = max([np.sum(y_train == i) for i in range(num_classes)])
x_train_balanced = []
y_train_balanced = []
for i in range(num_classes):
    class_x = x_train[y_train == i]
    class_y = y_train[y_train == i]
    if len(class_x) == 0:
        continue  # Theoretically, it should not occur.
    class_x_upsampled, class_y_upsampled = resample(
        class_x, class_y, replace=True, n_samples=max_count, random_state=42
    )
    x_train_balanced.append(class_x_upsampled)
    y_train_balanced.append(class_y_upsampled)
x_train = np.concatenate(x_train_balanced, axis=0)
y_train = np.concatenate(y_train_balanced, axis=0)

# ==== 7. Standardize the data ====
mean = x_train.mean(axis=(0, 1), keepdims=True)
std = x_train.std(axis=(0, 1), keepdims=True)
x_train = (x_train - mean) / (std + 1e-8)
x_test = (x_test - mean) / (std + 1e-8)

# ==== 8. Split into training/validation sets ====
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# ==== 9. One-hot encode the labels ====
y_train_cat = np.eye(num_classes)[y_train]
y_val_cat = np.eye(num_classes)[y_val]
y_test_cat = np.eye(num_classes)[y_test]

# ==== 10. Reshape Dense inputs ====
x_train = x_train.reshape(x_train.shape[0], -1)
x_val = x_val.reshape(x_val.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)

# Convert data to PyTorch tensors
x_train = torch.tensor(x_train, dtype=torch.float32)
y_train_cat = torch.tensor(y_train_cat, dtype=torch.float32)
x_val = torch.tensor(x_val, dtype=torch.float32)
y_val_cat = torch.tensor(y_val_cat, dtype=torch.float32)
x_test = torch.tensor(x_test, dtype=torch.float32)
y_test_cat = torch.tensor(y_test_cat, dtype=torch.float32)

# ==== 11. Build and train the model ====
class FCN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(FCN, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.dropout1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(128, 64)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(64, num_classes)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout1(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout2(x)
        x = torch.softmax(self.fc3(x), dim=1)
        return x

model = FCN(input_size=x_train.shape[1], num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# DataLoader for batching
train_dataset = TensorDataset(x_train, torch.tensor(y_train, dtype=torch.long))
val_dataset = TensorDataset(x_val, torch.tensor(y_val, dtype=torch.long))
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128)

# Training loop
epochs = 30
for epoch in range(epochs):
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    
    # Validation
    model.eval()
    val_loss = 0.0
    val_acc = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            val_acc += (outputs.argmax(dim=1) == batch_y).float().mean().item()
    val_loss /= len(val_loader)
    val_acc /= len(val_loader)
    print(f"Epoch {epoch+1}/{epochs} - Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

# ==== 12. Evaluate accuracy on test set ====
model.eval()
test_outputs = model(x_test)
test_acc = (test_outputs.argmax(dim=1) == torch.tensor(y_test)).float().mean().item()
print(f"Test accuracy: {test_acc:.4f}")

# ==== 13. Check for duplicates between training and test sets ====
print(np.intersect1d(x_train.numpy().reshape(x_train.shape[0], -1), x_test.numpy().reshape(x_test.shape[0], -1)).shape)
print("Train label dist:", np.bincount(y_train))
print("Test label dist:", np.bincount(y_test))

Train label dist: {0: 1798, 1: 273, 2: 227, 3: 236, 4: 225}
Test label dist: {0: 593, 1: 104, 2: 105, 3: 103, 4: 96}
GAN label dist: {0: 4053}
Epoch 1/30 - Val Loss: 0.9761, Val Acc: 0.9290
Epoch 2/30 - Val Loss: 0.9752, Val Acc: 0.9295
Epoch 3/30 - Val Loss: 0.9653, Val Acc: 0.9396
Epoch 4/30 - Val Loss: 0.9652, Val Acc: 0.9396
Epoch 5/30 - Val Loss: 0.9650, Val Acc: 0.9398
Epoch 6/30 - Val Loss: 0.9650, Val Acc: 0.9398
Epoch 7/30 - Val Loss: 0.9650, Val Acc: 0.9398
Epoch 8/30 - Val Loss: 0.9650, Val Acc: 0.9398
Epoch 9/30 - Val Loss: 0.9650, Val Acc: 0.9398
Epoch 10/30 - Val Loss: 0.9650, Val Acc: 0.9396
Epoch 11/30 - Val Loss: 0.9650, Val Acc: 0.9398
Epoch 12/30 - Val Loss: 0.9650, Val Acc: 0.9398
Epoch 13/30 - Val Loss: 0.9649, Val Acc: 0.9398
Epoch 14/30 - Val Loss: 0.9650, Val Acc: 0.9398
Epoch 15/30 - Val Loss: 0.9650, Val Acc: 0.9398
Epoch 16/30 - Val Loss: 0.9650, Val Acc: 0.9398
Epoch 17/30 - Val Loss: 0.9650, Val Acc: 0.9398
Epoch 18/30 - Val Loss: 0.9606, Val Acc: 0.9442
Ep

## FCN_gan_Split

In [2]:
import pickle
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

# ==== 1. load original data ====
with open('/kaggle/input/fsdfsd/train.pickle', 'rb') as f:
    train_data = pickle.load(f)
x_train = np.stack(train_data['sensor_data'])
y_train = np.array(train_data['label'])

# ==== 2. load GAN-generated data ====
with open('/kaggle/input/fsdfsd/generated_data.pickle', 'rb') as f:
    gan_data = pickle.load(f)
x_gan = np.array([item['sensor_data'] for item in gan_data])
y_gan = np.array([item['label'] for item in gan_data])

# ==== 3. load test set ====
with open('/kaggle/input/fsdfsd/test.pickle', 'rb') as f:
    test_data = pickle.load(f)
x_test = np.stack(test_data['sensor_data'])
y_test = np.array(test_data['label'])

# ==== 4. only keep categories which exist in test set ====
test_classes = set(np.unique(y_test))
mask_train = np.array([y in test_classes for y in y_train])
mask_gan = np.array([y in test_classes for y in y_gan])

x_train = x_train[mask_train]
y_train = y_train[mask_train]
x_gan = x_gan[mask_gan]
y_gan = y_gan[mask_gan]

# LabelEncoder only encodes the categories involved in the test set
all_classes = sorted(list(test_classes))
le = LabelEncoder()
le.fit(all_classes)
y_train = le.transform(y_train)
y_gan = le.transform(y_gan)
y_test = le.transform(y_test)

def print_label_stats(name, y):
    unique, counts = np.unique(y, return_counts=True)
    print(f"{name} label dist:", dict(zip(unique, counts)))
print_label_stats("Train", y_train)
print_label_stats("Test", y_test)
print_label_stats("GAN", y_gan)

# ==== 5. Merge GAN data ====
x_train = np.concatenate([x_train, x_gan], axis=0)
y_train = np.concatenate([y_train, y_gan], axis=0)

# ==== 6. Upsampling to achieve class balance ====
num_classes = len(all_classes)
max_count = max([np.sum(y_train == i) for i in range(num_classes)])
x_train_balanced = []
y_train_balanced = []
for i in range(num_classes):
    class_x = x_train[y_train == i]
    class_y = y_train[y_train == i]
    if len(class_x) == 0:
        continue  # Theoretically, it should not occur.
    class_x_upsampled, class_y_upsampled = resample(
        class_x, class_y, replace=True, n_samples=max_count, random_state=42
    )
    x_train_balanced.append(class_x_upsampled)
    y_train_balanced.append(class_y_upsampled)
x_train = np.concatenate(x_train_balanced, axis=0)
y_train = np.concatenate(y_train_balanced, axis=0)

# ==== 7. Standardize the data ====
mean = x_train.mean(axis=(0, 1), keepdims=True)
std = x_train.std(axis=(0, 1), keepdims=True)
x_train = (x_train - mean) / (std + 1e-8)
x_test = (x_test - mean) / (std + 1e-8)

# ==== 8. Reshape Dense inputs ====
x_train = x_train.reshape(x_train.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)

# Convert data to PyTorch tensors
x_train = torch.tensor(x_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
x_test = torch.tensor(x_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

# ==== 9. Define the model ====
class FCN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(FCN, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.dropout1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(128, 64)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(64, num_classes)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout1(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout2(x)
        x = torch.softmax(self.fc3(x), dim=1)
        return x

# ==== 10. Training with multiple random splits ====
def train_model_multiple_splits(model, dataset, criterion, optimizer, epochs, device, num_splits=5):
    best_val_accuracy = 0.0

    for split_index in range(num_splits):
        print(f"Training with split {split_index + 1}/{num_splits}")

        # Randomly split dataset into training and validation
        train_size = int(0.8 * len(dataset))
        val_size = len(dataset) - train_size
        train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

        train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

        model = model.to(device)
        for epoch in range(epochs):
            model.train()
            train_loss = 0.0

            for x_batch, y_batch in train_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                outputs = model(x_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            # Validation
            model.eval()
            val_loss = 0.0
            val_acc = 0.0
            with torch.no_grad():
                for x_batch, y_batch in val_loader:
                    x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                    outputs = model(x_batch)
                    loss = criterion(outputs, y_batch)
                    val_loss += loss.item()
                    val_acc += (outputs.argmax(dim=1) == y_batch).float().mean().item()
            val_loss /= len(val_loader)
            val_acc /= len(val_loader)

            print(f"Epoch {epoch+1}/{epochs} - Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

            # Track best validation accuracy
            if val_acc > best_val_accuracy:
                best_val_accuracy = val_acc

    print(f"Best validation accuracy across splits: {best_val_accuracy:.4f}")

# ==== 11. Prepare dataset and train ====
dataset = TensorDataset(x_train, y_train)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FCN(input_size=x_train.shape[1], num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_model_multiple_splits(model, dataset, criterion, optimizer, epochs=30, device=device, num_splits=5)

# ==== 12. Evaluate accuracy on test set ====
model.eval()
test_outputs = model(x_test.to(device))
test_acc = (test_outputs.argmax(dim=1) == y_test.to(device)).float().mean().item()
print(f"Test accuracy: {test_acc:.4f}")

Train label dist: {0: 1798, 1: 273, 2: 227, 3: 236, 4: 225}
Test label dist: {0: 593, 1: 104, 2: 105, 3: 103, 4: 96}
GAN label dist: {0: 4053}
Training with split 1/5
Epoch 1/30 - Val Loss: 0.9728, Val Acc: 0.9318
Epoch 2/30 - Val Loss: 0.9639, Val Acc: 0.9408
Epoch 3/30 - Val Loss: 0.9636, Val Acc: 0.9412
Epoch 4/30 - Val Loss: 0.9635, Val Acc: 0.9413
Epoch 5/30 - Val Loss: 0.9636, Val Acc: 0.9412
Epoch 6/30 - Val Loss: 0.9636, Val Acc: 0.9410
Epoch 7/30 - Val Loss: 0.9547, Val Acc: 0.9500
Epoch 8/30 - Val Loss: 0.9548, Val Acc: 0.9500
Epoch 9/30 - Val Loss: 0.9543, Val Acc: 0.9505
Epoch 10/30 - Val Loss: 0.9543, Val Acc: 0.9505
Epoch 11/30 - Val Loss: 0.9543, Val Acc: 0.9505
Epoch 12/30 - Val Loss: 0.9543, Val Acc: 0.9505
Epoch 13/30 - Val Loss: 0.9543, Val Acc: 0.9505
Epoch 14/30 - Val Loss: 0.9545, Val Acc: 0.9503
Epoch 15/30 - Val Loss: 0.9543, Val Acc: 0.9505
Epoch 16/30 - Val Loss: 0.9543, Val Acc: 0.9505
Epoch 17/30 - Val Loss: 0.9543, Val Acc: 0.9505
Epoch 18/30 - Val Loss: 0.

## two_class_label_split

In [3]:
import pickle
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ==== 1. load original data ====
with open('/kaggle/input/fsdfsd/train.pickle', 'rb') as f:
    train_data = pickle.load(f)
x_train = np.stack(train_data['sensor_data'])
y_train = np.array(train_data['label'])

# ==== 2. load GAN-generated data ====
with open('/kaggle/input/fsdfsd/generated_data.pickle', 'rb') as f:
    gan_data = pickle.load(f)
x_gan = np.array([item['sensor_data'] for item in gan_data])
y_gan = np.array([item['label'] for item in gan_data])

# ==== 3. load test set ====
with open('/kaggle/input/fsdfsd/test.pickle', 'rb') as f:
    test_data = pickle.load(f)
x_test = np.stack(test_data['sensor_data'])
y_test = np.array(test_data['label'])

# ==== 4. Remap labels to two categories: working (0) and non-working (1) ====
def remap_labels(y):
    return np.array([0 if label == 0000 else 1 for label in y])

y_train = remap_labels(y_train)
y_gan = remap_labels(y_gan)
y_test = remap_labels(y_test)

def print_label_stats(name, y):
    unique, counts = np.unique(y, return_counts=True)
    print(f"{name} label distribution:", dict(zip(unique, counts)))

print_label_stats("Train", y_train)
print_label_stats("Test", y_test)
print_label_stats("GAN", y_gan)

# ==== 5. Merge GAN data ====
x_train = np.concatenate([x_train, x_gan], axis=0)
y_train = np.concatenate([y_train, y_gan], axis=0)

# ==== 6. Upsampling to achieve class balance ====
num_classes = 2
max_count = max([np.sum(y_train == i) for i in range(num_classes)])
x_train_balanced = []
y_train_balanced = []
for i in range(num_classes):
    class_x = x_train[y_train == i]
    class_y = y_train[y_train == i]
    if len(class_x) == 0:
        continue  # Theoretically, it should not occur.
    class_x_upsampled, class_y_upsampled = resample(
        class_x, class_y, replace=True, n_samples=max_count, random_state=42
    )
    x_train_balanced.append(class_x_upsampled)
    y_train_balanced.append(class_y_upsampled)
x_train = np.concatenate(x_train_balanced, axis=0)
y_train = np.concatenate(y_train_balanced, axis=0)

# ==== 7. Standardize the data ====
mean = x_train.mean(axis=(0, 1), keepdims=True)
std = x_train.std(axis=(0, 1), keepdims=True)
x_train = (x_train - mean) / (std + 1e-8)
x_test = (x_test - mean) / (std + 1e-8)

# ==== 8. Split into training/validation sets ====
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# ==== 9. One-hot encode the labels ====
y_train_cat = np.eye(num_classes)[y_train]
y_val_cat = np.eye(num_classes)[y_val]
y_test_cat = np.eye(num_classes)[y_test]

# ==== 10. Reshape Dense inputs ====
x_train = x_train.reshape(x_train.shape[0], -1)
x_val = x_val.reshape(x_val.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)

# Convert data to PyTorch tensors
x_train = torch.tensor(x_train, dtype=torch.float32)
y_train_cat = torch.tensor(y_train_cat, dtype=torch.float32)
x_val = torch.tensor(x_val, dtype=torch.float32)
y_val_cat = torch.tensor(y_val_cat, dtype=torch.float32)
x_test = torch.tensor(x_test, dtype=torch.float32)
y_test_cat = torch.tensor(y_test_cat, dtype=torch.float32)

# ==== 11. Build and train the model ====
class FCN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(FCN, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.dropout1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(128, 64)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(64, num_classes)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout1(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout2(x)
        x = torch.softmax(self.fc3(x), dim=1)
        return x

model = FCN(input_size=x_train.shape[1], num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# DataLoader for batching
train_dataset = TensorDataset(x_train, torch.tensor(y_train, dtype=torch.long))
val_dataset = TensorDataset(x_val, torch.tensor(y_val, dtype=torch.long))
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128)

# Training loop
epochs = 30
for epoch in range(epochs):
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    
    # Validation
    model.eval()
    val_loss = 0.0
    val_acc = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            val_acc += (outputs.argmax(dim=1) == batch_y).float().mean().item()
    val_loss /= len(val_loader)
    val_acc /= len(val_loader)
    print(f"Epoch {epoch+1}/{epochs} - Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

# ==== 12. Evaluate accuracy on test set ====
model.eval()
test_outputs = model(x_test)
test_acc = (test_outputs.argmax(dim=1) == torch.tensor(y_test)).float().mean().item()
print(f"Test accuracy: {test_acc:.4f}")

# ==== 13. Check for duplicates between training and test sets ====
print(np.intersect1d(x_train.numpy().reshape(x_train.shape[0], -1), x_test.numpy().reshape(x_test.shape[0], -1)).shape)
print("Train label dist:", np.bincount(y_train))
print("Test label dist:", np.bincount(y_test))

Train label distribution: {0: 1798, 1: 2255}
Test label distribution: {0: 593, 1: 408}
GAN label distribution: {0: 4053}
Epoch 1/30 - Val Loss: 0.5503, Val Acc: 0.7578
Epoch 2/30 - Val Loss: 0.5423, Val Acc: 0.7707
Epoch 3/30 - Val Loss: 0.5405, Val Acc: 0.7697
Epoch 4/30 - Val Loss: 0.5352, Val Acc: 0.7785
Epoch 5/30 - Val Loss: 0.5337, Val Acc: 0.7797
Epoch 6/30 - Val Loss: 0.5358, Val Acc: 0.7775
Epoch 7/30 - Val Loss: 0.5357, Val Acc: 0.7762
Epoch 8/30 - Val Loss: 0.5362, Val Acc: 0.7754
Epoch 9/30 - Val Loss: 0.5339, Val Acc: 0.7783
Epoch 10/30 - Val Loss: 0.5321, Val Acc: 0.7807
Epoch 11/30 - Val Loss: 0.5316, Val Acc: 0.7818
Epoch 12/30 - Val Loss: 0.5300, Val Acc: 0.7824
Epoch 13/30 - Val Loss: 0.5309, Val Acc: 0.7820
Epoch 14/30 - Val Loss: 0.5307, Val Acc: 0.7812
Epoch 15/30 - Val Loss: 0.5296, Val Acc: 0.7834
Epoch 16/30 - Val Loss: 0.5290, Val Acc: 0.7842
Epoch 17/30 - Val Loss: 0.5300, Val Acc: 0.7822
Epoch 18/30 - Val Loss: 0.5297, Val Acc: 0.7830
Epoch 19/30 - Val Loss: 